[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · **[07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb)** · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 6 课：Human-in-the-loop（interrupt）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) ｜ [➡️ 下一课：第 8 课：流式输出（v2 / stream_events v3）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb)

---

# 第 7 课：时间旅行 —— Replay（重放）与 Fork（分叉）

因为每个超步都有检查点，你可以：
- Replay：用历史上某个检查点的 config 调用 invoke(None, config)，
  该检查点之前的步骤不再执行，从那里继续往后跑。
- Fork：用 update_state(历史config, 新值) 在历史点上创建一个"分支"检查点，
  再从这个分支继续运行 —— 原历史保持不变。

In [1]:
#@title 🔧 第 0 步：初始化环境（在 Colab 中点击左侧 ▶ 即可自动安装依赖，本地运行自动跳过） { display-mode: "form" }
import importlib.util, subprocess, sys
if importlib.util.find_spec('langgraph') is None:
    print('⏳ 正在安装 LangGraph 环境（约 10 秒）...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph', 'langchain-core', 'langgraph-checkpoint-sqlite'])
    print('✅ 安装完成！')
else:
    import importlib.metadata as m
    print(f'✅ 环境已就绪：langgraph={m.version("langgraph")}')


✅ 环境已就绪：langgraph=1.2.22


In [2]:
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph


class State(TypedDict):
    topic: str
    draft: str
    final: str


def write(state: State):
    print(f"    [write] 主题 = {state['topic']}")
    return {"draft": f"一篇关于『{state['topic']}』的草稿"}


def polish(state: State):
    print("    [polish]")
    return {"final": state["draft"] + " ✨(已润色)"}


graph = (
    StateGraph(State)
    .add_node(write).add_node(polish)
    .add_edge(START, "write").add_edge("write", "polish")
    .compile(checkpointer=InMemorySaver())
)

## ▶️ 运行实验

In [3]:
cfg = {"configurable": {"thread_id": "tt"}}
print("== 原始运行 ==")
print(graph.invoke({"topic": "猫"}, cfg)["final"])

history = list(graph.get_state_history(cfg))
for s in history:
    print(f"  step={s.metadata['step']:>2} next={s.next} values={s.values}")

# 找到 "即将执行 write" 的检查点（write 之前）
before_write = next(s for s in history if s.next == ("write",))

print("\n== Replay：从 write 之前重放 ==")
print(graph.invoke(None, before_write.config)["final"])

print("\n== Fork：把 topic 改成 '狗'，从分支继续 ==")
fork_cfg = graph.update_state(before_write.config, {"topic": "狗"})
print(graph.invoke(None, fork_cfg)["final"])

print("\n最新状态:", graph.get_state(cfg).values["final"])
print("历史检查点总数:", len(list(graph.get_state_history(cfg))))

== 原始运行 ==
    [write] 主题 = 猫
    [polish]
一篇关于『猫』的草稿 ✨(已润色)
  step= 2 next=() values={'topic': '猫', 'draft': '一篇关于『猫』的草稿', 'final': '一篇关于『猫』的草稿 ✨(已润色)'}
  step= 1 next=('polish',) values={'topic': '猫', 'draft': '一篇关于『猫』的草稿'}
  step= 0 next=('write',) values={'topic': '猫'}
  step=-1 next=('__start__',) values={}

== Replay：从 write 之前重放 ==
    [write] 主题 = 猫
    [polish]
一篇关于『猫』的草稿 ✨(已润色)

== Fork：把 topic 改成 '狗'，从分支继续 ==
    [write] 主题 = 狗
    [polish]
一篇关于『狗』的草稿 ✨(已润色)

最新状态: 一篇关于『狗』的草稿 ✨(已润色)
历史检查点总数: 10


## ✏️ 练习：
 1. 找到 "即将执行 polish" 的检查点，fork 时把 draft 改掉，观察 write 是否会重跑。
 2. update_state 有一个 as_node 参数，试着理解它的作用（决定"下一个节点"是谁）。

In [4]:
# 在这里写你的练习代码 👇


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · **[07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb)** · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 6 课：Human-in-the-loop（interrupt）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) ｜ [➡️ 下一课：第 8 课：流式输出（v2 / stream_events v3）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb)

---